In [1]:
pip install --no-cache-dir numpy==2.0.2

In [2]:
!pip install --no-cache-dir mediapipe==0.10.20

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 101.4 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of jax to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 76.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 274.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 358.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 241.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 219.3 MB/s eta 0:00:0

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp "/content/drive/MyDrive/Colab Notebooks/Integration/"*.ipynb /content/

In [3]:
!ls

Arm_and_Space_positions.ipynb  movement1_prava.ipynb
contact_types_e.ipynb	       Movement_2.ipynb
drive			       ori_model2.ipynb
fing_locations_d.ipynb	       sample_data
hand_location_video_P.ipynb    Sravs.ipynb
Handshape_Model.ipynb	       upper_body_locations_video.ipynb
Head_and_face_location.ipynb


In [4]:
!jupyter nbconvert --to python Handshape_Model.ipynb
!jupyter nbconvert --to python ori_model2.ipynb
!jupyter nbconvert --to python upper_body_locations_video.ipynb
!jupyter nbconvert --to python Head_and_face_location.ipynb
!jupyter nbconvert --to python hand_location_video_P.ipynb
!jupyter nbconvert --to python fing_locations_d.ipynb
!jupyter nbconvert --to python contact_types_e.ipynb
!jupyter nbconvert --to python Arm_and_Space_positions.ipynb
!jupyter nbconvert --to python movement1_prava.ipynb
!jupyter nbconvert --to python Movement_2.ipynb

[NbConvertApp] Converting notebook Handshape_Model.ipynb to python
[NbConvertApp] Writing 3294 bytes to Handshape_Model.py
[NbConvertApp] Converting notebook ori_model2.ipynb to python
[NbConvertApp] Writing 13060 bytes to ori_model2.py
[NbConvertApp] Converting notebook upper_body_locations_video.ipynb to python
[NbConvertApp] Writing 10131 bytes to upper_body_locations_video.py
[NbConvertApp] Converting notebook Head_and_face_location.ipynb to python
[NbConvertApp] Writing 8825 bytes to Head_and_face_location.py
[NbConvertApp] Converting notebook hand_location_video_P.ipynb to python
[NbConvertApp] Writing 5945 bytes to hand_location_video_P.py
[NbConvertApp] Converting notebook fing_locations_d.ipynb to python
[NbConvertApp] Writing 7726 bytes to fing_locations_d.py
[NbConvertApp] Converting notebook contact_types_e.ipynb to python
[NbConvertApp] Writing 6803 bytes to contact_types_e.py
[NbConvertApp] Converting notebook Arm_and_Space_positions.ipynb to python
[NbConvertApp] Writing

In [5]:

import sys
sys.path.append('/content')

In [6]:
import numpy as np
import cv2
import mediapipe as mp

In [7]:
from Handshape_Model import run_handshape_module
from ori_model2 import run_orientation_module
from upper_body_locations_video import run_upper_body_location_module
from Head_and_face_location import run_head_face_location_module
from hand_location_video_P import run_hand_location_module
from fing_locations_d import run_finger_location_module
from contact_types_e import run_contact_type_module
from Arm_and_Space_positions import run_arm_space_module

from movement1_prava import run_movement1_module
from Movement_2 import run_movement2_module

Frames with hand detected: 0
Prediction counts: Counter()
FINAL LABEL: no prediction
cap.isOpened(): False
First frame read: False


In [8]:
mp_pose = mp.solutions.pose

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

In [9]:
video_path = "/content/drive/MyDrive/Colab Notebooks/Integration/Prompt_1.mp4"
output_video ="/content/drive/MyDrive/Colab Notebooks/Integration/output_prva.mp4"

In [10]:

print("Running modules...")

handshape = run_handshape_module(video_path)
orientation = run_orientation_module(video_path)

upper_body = run_upper_body_location_module(video_path)
head_face = run_head_face_location_module(video_path)

hand_location = run_hand_location_module(video_path)
finger_location = run_finger_location_module(video_path)

contact = run_contact_type_module(video_path)
arm_space = run_arm_space_module(video_path)

movement1 = run_movement1_module(video_path)
movement2 = run_movement2_module(video_path)

Running modules...


In [11]:
def extract_labels(module_output):
    """
    Extract useful HamNoSys labels from module outputs
    """

    if isinstance(module_output, dict):

        # Most modules store labels here
        if "per_frame" in module_output:
            labels = module_output["per_frame"]

            # Flatten tuples and remove None
            cleaned = []
            for item in labels:

                if item is None:
                    continue

                if isinstance(item, tuple):
                    cleaned.extend([str(x) for x in item if x is not None])
                else:
                    cleaned.append(str(item))

            return cleaned

    # If already string
    if isinstance(module_output, str):
        return [module_output]

    return []

In [12]:
def combine_hamnosys(*modules):

    final = []

    for m in modules:
        final.extend(extract_labels(m))

    # Remove duplicates but keep order
    seen = set()
    clean = []

    for x in final:
        if x not in seen:
            clean.append(x)
            seen.add(x)

    return " ".join(clean)

In [13]:
def generate_hamnosys(video_path):

    '''
    handshape       = run_handshape_module(video_path)
    orientation     = run_orientation_module(video_path)
    upper_body      = run_upper_body_location_module(video_path)
    head_face       = run_head_face_location_module(video_path)
    hand_location   = run_hand_location_module(video_path)
    finger_location = run_finger_location_module(video_path)
    contact         = run_contact_type_module(video_path)
    arm_space       = run_arm_space_module(video_path)
    movement1       = run_movement1_module(video_path)
    movement2       = run_movement2_module(video_path)
    '''
    print("\n===== MODULE OUTPUTS =====")
    print("Handshape       :", handshape)
    print("Orientation     :", orientation)
    print("Arm & Space     :", arm_space)
    print("Upper Body      :", upper_body)
    print("Head & Face     :", head_face)
    print("Hand Location   :", hand_location)
    print("Finger Location :", finger_location)
    print("Contact Type    :", contact)
    print("Movement 1      :", movement1)
    print("Movement 2      :", movement2)


    hamnosys_code = combine_hamnosys(
        handshape,
        orientation,
        upper_body,
        head_face,
        hand_location,
        finger_location,
        contact,
        arm_space,
        movement1,
        movement2
    )

    print("\n========== FINAL HAMNOSYS ==========")
    print(hamnosys_code)

    return hamnosys_code

In [14]:
hamnosys_output = generate_hamnosys(video_path)


===== MODULE OUTPUTS =====
Handshape       : {'per_frame': ['hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamthumb', 'hamthumb', 'hamthumb', 'hamthumb', 'hamthumb', 'hamflathand', 'hamflathand', 'hamthumb', 'hamflathand', 'hamflathand', 'hamthumb', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 'hamflathand', 

In [ ]:
#output_video ="/content/drive/MyDrive/Colab Notebooks/Integration/output_pi.mp4"

In [15]:
# =============================
# CREATE VIDEO WITH ALL MODULE OUTPUTS
# =============================

import cv2

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Cannot open video")

ret, frame = cap.read()
h, w, _ = frame.shape

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(output_video, fourcc, 25, (w, h))

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))


# =============================
# SAFE LABEL
# =============================

def safe(x):
    if x is None:
        return "none"
    return str(x)


# =============================
# FRAME EXTRACTOR
# =============================

def get_frames(output, total_frames):

    if isinstance(output, dict):
        arr = output.get("per_frame", [])
        if len(arr) > 0:
            return arr

    if isinstance(output, str):
        return [output] * total_frames

    if isinstance(output, list):
        return output

    return ["none"] * total_frames


# =============================
# EXTRACT FRAME DATA
# =============================

handshape_frames = get_frames(handshape, total_frames)
orientation_frames = get_frames(orientation, total_frames)

upper_frames = get_frames(upper_body, total_frames)
head_frames = get_frames(head_face, total_frames)
hand_frames = get_frames(hand_location, total_frames)
finger_frames = get_frames(finger_location, total_frames)
contact_frames = get_frames(contact, total_frames)
arm_frames = get_frames(arm_space, total_frames)

movement1_frames = get_frames(movement1, total_frames)
movement2_frames = get_frames(movement2, total_frames)


# =============================
# VIDEO PROCESSING
# =============================

frame_id = 0
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

while cap.isOpened():

    ret, frame = cap.read()
    if not ret:
        break

    # ---------- FRAME LABELS ----------

    handshape_label = safe(handshape_frames[frame_id])
    orientation_label = safe(orientation_frames[frame_id])

    # combine ALL location modules
    location_label = " ".join([
        safe(upper_frames[frame_id]),
        safe(head_frames[frame_id]),
        safe(hand_frames[frame_id]),
        safe(finger_frames[frame_id]),
        safe(contact_frames[frame_id]),
        safe(arm_frames[frame_id])
    ])

    movement_label = " ".join([
        safe(movement1_frames[frame_id]),
        safe(movement2_frames[frame_id])
    ])


    # =============================
    # DRAW TEXT
    # =============================

    cv2.putText(frame,
                f"Handshape: {handshape_label}",
                (30,40),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2)

    cv2.putText(frame,
                f"Orientation: {orientation_label}",
                (30,70),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2)

    cv2.putText(frame,
                f"Location: {location_label}",
                (30,110),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2)

    cv2.putText(frame,
                f"Movement: {movement_label}",
                (30,150),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0,255,0),
                2)

    out.write(frame)

    frame_id += 1


cap.release()
out.release()

print("Video saved:", output_video)

Video saved: /content/drive/MyDrive/Colab Notebooks/Integration/output_prva.mp4


In [ ]:
def process_video(video_path):

    hamnosys = generate_hamnosys(video_path)

    results = {
        "hamnosys": hamnosys,
        "output_video": "/content/output_video.mp4"
    }

    return results

In [16]:
!pip install gradio opencv-python mediapipe

INFO: pip is looking at multiple versions of mediapipe to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 51.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
  Attempting uninstall: mediapipe
    Found existing installation: mediapipe 0.10.20
    Uninstalling mediapipe-0.10.20:
      Successfully uninstalled mediapipe-0.10.20
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [1]:
import gradio as gr
import tempfile
import shutil
import os

# import your pipeline
from integration_pipeline import generate_hamnosys

def run_pipeline(video):

    temp_video = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4").name
    shutil.copy(video, temp_video)

    # run hamnosys pipeline
    hamnosys = generate_hamnosys(temp_video)

    output_video = "output_video.mp4"

    sigml = f"""
<sigml>
 <hamnosys_nonmanual></hamnosys_nonmanual>

 <sign>
  <hamnosys_manual>
   {hamnosys}
  </hamnosys_manual>
 </sign>

</sigml>
"""

    with open("output.sigml","w") as f:
        f.write(sigml)

    return output_video, hamnosys, sigml


app = gr.Interface(

    fn=run_pipeline,

    inputs=gr.Video(label="Upload Sign Language Video"),

    outputs=[
        gr.Video(label="Processed Output Video"),
        gr.Textbox(label="HamNoSys Output"),
        gr.Textbox(label="SiGML Output")
    ],

    title="Sign Language → HamNoSys Generator",
    description="Upload a sign language video to generate HamNoSys and SiGML"
)

app.launch()

ModuleNotFoundError: No module named 'integration_pipeline'